In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib

import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

from functions.read_g4bl_data import readTraceData, readDetData
from functions.set_plot_settings import setPlotSettings
from functions.get_transfer_matrix import getInitVals, getFinVals, getTransferMatrix

setPlotSettings(font=True)

## Calculate one-turn transfer matrix

In [2]:
### Read in data from 12 tracking files:

data_dx_pos = readDetData('zntuple_1turn_x_pos.txt')
data_ref = data_dx_pos[data_dx_pos['EventID']==-1] # same for all datasets
data_dx_pos = data_dx_pos[data_dx_pos['EventID']==1]
data_dx_neg = readDetData('zntuple_1turn_x_neg.txt')
data_dx_neg = data_dx_neg[data_dx_neg['EventID']==1]

data_dy_pos = readDetData('zntuple_1turn_y_pos.txt')
data_dy_pos = data_dy_pos[data_dy_pos['EventID']==1]
data_dy_neg = readDetData('zntuple_1turn_y_neg.txt')
data_dy_neg = data_dy_neg[data_dy_neg['EventID']==1]

data_dxp_pos = readDetData('zntuple_1turn_xp_pos.txt')
data_dxp_pos = data_dxp_pos[data_dxp_pos['EventID']==1]
data_dxp_neg = readDetData('zntuple_1turn_xp_neg.txt')
data_dxp_neg = data_dxp_neg[data_dxp_neg['EventID']==1]

data_dyp_pos = readDetData('zntuple_1turn_yp_pos.txt')
data_dyp_pos = data_dyp_pos[data_dyp_pos['EventID']==1]
data_dyp_neg = readDetData('zntuple_1turn_yp_neg.txt')
data_dyp_neg = data_dyp_neg[data_dyp_neg['EventID']==1]

data_dt_pos = readDetData('zntuple_1turn_t_pos.txt')
data_dt_pos = data_dt_pos[data_dt_pos['EventID']==1]
data_dt_neg = readDetData('zntuple_1turn_t_neg.txt')
data_dt_neg = data_dt_neg[data_dt_neg['EventID']==1]

data_dp_pos = readDetData('zntuple_1turn_p_pos.txt')
data_dp_pos = data_dp_pos[data_dp_pos['EventID']==1]
data_dp_neg = readDetData('zntuple_1turn_p_neg.txt')
data_dp_neg = data_dp_neg[data_dp_neg['EventID']==1]

In [3]:
### Print initial offsets:

print('------------- x [mm] -------------')
print('reference: '+ str(data_ref['x'].values[0]))
print('+ offset: '+ str(data_dx_pos['x'].values[0]))
print('- offset: '+ str(data_dx_neg['x'].values[0]))

print('------------- y [mm] -------------')
print('reference: '+ str(data_ref['y'].values[0]))
print('+ offset: '+ str(data_dy_pos['y'].values[0]))
print('- offset: '+ str(data_dy_neg['y'].values[0]))

print('------------- xp [rad] -------------')
print('reference: '+ str(data_ref['xp'].values[0]))
print('+ offset: '+ str(data_dxp_pos['xp'].values[0]))
print('- offset: '+ str(data_dxp_neg['xp'].values[0]))

print('------------- yp [rad] -------------')
print('reference: '+ str(data_ref['yp'].values[0]))
print('+ offset: '+ str(data_dyp_pos['yp'].values[0]))
print('- offset: '+ str(data_dyp_neg['yp'].values[0]))

print('------------- t [ns] -------------')
print('reference: '+ str(data_ref['t'].values[0]))
print('+ offset: '+ str(data_dt_pos['t'].values[0]-data_ref['t'].values[0]))
print('- offset: '+ str(data_dt_neg['t'].values[0]-data_ref['t'].values[0]))

print('------------- delta -------------')
print('reference: '+ str((data_ref['ptotal'].values[0]-data_ref['ptotal'].values[0])/data_ref['ptotal'].values[0]))
print('+ offset: '+ str((data_dp_pos['ptotal'].values[0]-data_ref['ptotal'].values[0])/data_ref['ptotal'].values[0]))
print('- offset: '+ str((data_dp_neg['ptotal'].values[0]-data_ref['ptotal'].values[0])/data_ref['ptotal'].values[0]))

------------- x [mm] -------------
reference: 9.47954
+ offset: 9.48902
- offset: 9.47006
------------- y [mm] -------------
reference: 16.0791
+ offset: 16.0952
- offset: 16.063
------------- xp [rad] -------------
reference: 0.06414325251188256
+ offset: 0.06420724889822563
- offset: 0.06407875539600807
------------- yp [rad] -------------
reference: -0.037250015041212926
+ offset: -0.03728721697451016
- offset: -0.037212813107915695
------------- t [ns] -------------
reference: 0.0
+ offset: 0.00038
- offset: -0.00038
------------- delta -------------
reference: 0.0
+ offset: 0.001002702665798569
- offset: -0.0009977483459439325


In [4]:
### Calculate 6D transfer matrix:

M = getTransferMatrix(data_ref, data_dx_pos, data_dx_neg, data_dxp_pos, 
    data_dxp_neg, data_dy_pos, data_dy_neg, data_dyp_pos, data_dyp_neg, 
    data_dt_pos, data_dt_neg, data_dp_pos, data_dp_neg)

In [5]:
### Print 6D transfer matrix:
for i in range(len(M)):
    print(M[i])

[ 5.16350211e-01  5.48977176e+02 -1.34658385e+00  2.15042588e+00
  0.00000000e+00 -2.54792543e+01]
[1.50729454e-03 4.57647761e-01 3.11412276e-05 1.35204948e+00
 0.00000000e+00 3.62585959e-01]
[ 1.24472574e+00 -1.08954926e+01  5.09316770e-01  5.45670566e+02
  0.00000000e+00  8.90299232e+01]
[-2.22127617e-04 -1.36582514e+00  1.95255497e-03  4.91451447e-01
  0.00000000e+00  1.84553804e-01]
[ 0.          0.77824947  0.         -0.          0.92105263 -3.44922218]
[ 1.04626212e-04  1.35023150e-03 -6.96202370e-05  1.13651271e-03
  0.00000000e+00  9.98912282e-01]


In [8]:
### Testing -- print initial 6D coordinates of reference:
vi = getInitVals(data_ref, data_ref)
for i in range(len(vi)):
        print(vi[i])

[0.]
[0.]
[0.]
[0.]
[0.]
[0.]


In [9]:
### Testing -- print final 6D coordinates of reference according to M:
vf_calc = np.dot(M, vi)
for i in range(len(vf_calc)):
        print(vf_calc[i])

[0.]
[0.]
[0.]
[0.]
[0.]
[0.]


In [10]:
### Testing -- print final 6D coordinates of reference according to tracking:
vf_exp = getFinVals(data_ref, data_ref)
for i in range(len(vf_exp)):
        print(vf_exp[i])

[0.]
[0.]
[0.]
[0.]
[0.]
[0.]


In [12]:
### Testing -- print initial 6D coordinates of a displaced particle:
vi = getInitVals(data_dx_pos, data_ref)
for i in range(len(vi)):
        print(vi[i])

[0.00948]
[0.]
[0.]
[0.]
[0.]
[0.]


In [13]:
### Testing -- print final 6D coordinates of displaced particle according to M:
vf_calc = np.dot(M, vi)
for i in range(len(vf_calc)):
        print(vf_calc[i])

[0.004895]
[1.42891523e-05]
[0.0118]
[-2.10576981e-06]
[0.]
[9.91856492e-07]


In [14]:
### Testing -- print final 6D coordinates of displaced particle according to tracking:
vf_exp = getFinVals(data_dx_pos, data_ref)
for i in range(len(vf_exp)):
        print(vf_exp[i])

[0.00472]
[1.85508293e-05]
[0.0128]
[1.00274753e-07]
[0.]
[1.18381364e-06]


In [19]:
### Testing -- print final 6D coordinates of displaced particle according to M:
vf_calc = np.dot(M, vi)
for i in range(len(vf_calc)):
        # print(vf_calc[i])
        print(np.round(vf_calc[i],3))

[0.005]
[0.]
[0.012]
[-0.]
[0.]
[0.]


In [20]:
### Testing -- print final 6D coordinates of displaced particle according to tracking:
vf_exp = getFinVals(data_dx_pos, data_ref)
for i in range(len(vf_exp)):
        # print(vf_exp[i])
        print(np.round(vf_exp[i],3))

[0.005]
[0.]
[0.013]
[0.]
[0.]
[0.]


In [22]:
### Calculate determinant of M:
print(np.linalg.det(M))

1.0330978189210434


Symplectic ==> det(M) = 1, which we observe